In [ ]:
# clear all the parameters
dbutils.widgets.removeAll()

In [ ]:
%run "../utils/setup_databricks_session"

In [ ]:
from pyspark.errors import AnalysisException

from core.configs import WIDGETS_CONFIG, get_data_engine_entity_configs
from core.engine import deploy_ddl_waves, get_entity_ddls
from core.databricks_session import create_required_widgets, fetch_widget_values, log

In [ ]:
# define the required widgets
required_widgets = [
    "execution_mode",
    "silver_catalog",
    "silver_schema",
    "gold_catalog",
    "gold_schema",
    "target_entities",
]

# Create widgets for required parameters
create_required_widgets(dbutils, WIDGETS_CONFIG, required_widgets)

In [ ]:
# Get widget values and add any required params
params = fetch_widget_values(dbutils, print_validation=False)
log(f"Execution Mode: {params.execution_mode}")

In [ ]:
# Define target entity paths (Single Source of Truth)
target_sample_dim = f"{params.silver_catalog}.{params.silver_schema}.sample_dim"
target_sample_view = f"{params.gold_catalog}.{params.gold_schema}.sample_view"

print(f"[1/4] Adding new column on {target_sample_dim}...")
try:
    spark.sql(f"""
        ALTER TABLE {target_sample_dim}
        ADD COLUMN sample_new_col STRING;
    """)
except AnalysisException:
    print("[SKIPPED] Column was already added or table altered.")

print(f"[2/4] Enabling column mapping on {target_sample_view}...")
try:
    spark.sql(f"""
        ALTER VIEW {target_sample_view}
        WITH SCHEMA EVOLUTION;
    """)
except AnalysisException:
    print("[SKIPPED] Column mapping already configured or view altered.")

print(f"[3/4] renaming table from {target_sample_dim}...")
try:
    spark.sql(f"""
        ALTER TABLE {target_sample_dim}
        RENAME TO new_table_name;
    """)
except AnalysisException:
    print("[SKIPPED] table was already renamed or table altered.")

print(f"[4/4] renaming view from {target_sample_view}...")
try:
    spark.sql(f"""
        ALTER VIEW {target_sample_view}
        RENAME TO new_view_name;
    """)
except AnalysisException:
    print("[SKIPPED] view was already renamed or view altered.")


print("\n✅ Schema evolution completed successfully.")

In [ ]:
# Get DDLs for target entities
entity_ddls = get_entity_ddls(
    get_data_engine_entity_configs(),
    params.target_entities,
    params.silver_catalog,
    params.silver_schema,
    params.gold_catalog,
    params.gold_schema,
)
display(entity_ddls)

In [ ]:
# Deploy DDLs in parallel
deploy_ddl_waves(spark, entity_ddls, max_workers=8)